# Chapter 6a — Retrieval-Augmented Generation (Hotels)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzafarooq/advanced-rag-from-scratch/blob/main/colab_original_notebooks/Chapter_6a.ipynb)

Companion code for the **first half of Chapter 6** of *Build an Advanced RAG Application (From Scratch)*.

We connect the FAISS retrieval from Chapter 4 to the LLM prompting from Chapter 5 to produce a complete, grounded RAG pipeline:

1. **Retrieve** the most relevant hotel reviews for the user's query.
2. **Augment** the prompt with the retrieved context.
3. **Generate** a Markdown answer with inline citations.

We then graduate from FAISS (vectors live in process memory) to **Qdrant** — a real vector database that supports payloads and metadata filtering (e.g. *only Istanbul hotels*).



## 1. Setup

Needs `OPEN_ROUTER_API_KEY` in your `colab secrets` (sign up at https://openrouter.ai).


In [ ]:
from IPython.display import HTML, display

def set_css(*args, **kwargs):
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)


In [ ]:
!pip install huggingface
!pip install -U datasets
#!pip install sentence-transformers #uncomment only if needed
!pip install faiss-cpu
!pip install einops

# 2. Build the index (Paris-only, FAISS)

Same setup as Chapter 4 — we just keep the index alive so we can query it from RAG.


In [ ]:
import pandas as pd
from datasets import load_dataset
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
import scipy.spatial
from datasets import load_dataset



dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")

In [ ]:
df=pd.DataFrame(dataset['train'])
df.head()
df_paris = df.loc[df.locality=='Paris']
df_paris.drop_duplicates()
reviews = df_paris['review_text'].tolist()

In [ ]:
# Define helper functions to embed reviews and search the FAISS index.

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

def get_embeddings(data, model):
    """
    Generates embeddings for a list of text data.

    Args:
        data (list): A list of strings.
        model (SentenceTransformer): The pre-trained sentence transformer model.

    Returns:
        np.ndarray: The embeddings of the input data.
    """
    embeddings = model.encode(data, show_progress_bar=False).astype('float32')
    return embeddings

def search_faiss_index(query_embedding, faiss_index, k=5):
    """
    Performs a semantic search on a FAISS index using a query embedding.

    Args:
        query_embedding (np.ndarray): The embedding of the query string.
        faiss_index (faiss.IndexFlatIP): The FAISS index object.
        k (int): The number of nearest neighbors to retrieve.

    Returns:
        tuple: A tuple containing:
            - distances (np.ndarray): The distances of the retrieved neighbors.
            - indices (np.ndarray): The indices of the retrieved neighbors in the original data.
    """
    # Normalize the query embedding
    query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding)
    distances, indices = faiss_index.search(query_embedding_normalized, k)
    return distances, indices

def create_faiss_index(embeddings):
    """
    Creates a FAISS index from a set of embeddings.

    Args:
        embeddings (np.ndarray): The embeddings to index.

    Returns:
        faiss.IndexFlatIP: The created FAISS index object.
    """
    # Normalize the embeddings
    embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    # Initialize the FAISS index with cosine similarity (Inner Product)
    index = faiss.IndexFlatIP(embeddings_normalized.shape[1])
    # Add the normalized embeddings to the index
    index.add(embeddings_normalized)
    return index

# Example usage (assuming 'reviews' and 'model' are defined from previous code):
reviews = df_paris['review_text'].tolist()
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
if torch.cuda.is_available():
    model = model.to('cuda')

review_embeddings = get_embeddings(reviews, model)
faiss_index = create_faiss_index(review_embeddings)



# 3. Retrieve only — what FAISS hands to the LLM

Before we generate, look at the raw retrieval. RAG quality is bounded above by retrieval quality.


In [ ]:
query = "Hotel with a view of the Eiffel tower."
query_embedding = get_embeddings([query], model)

k = 25
distances, indices = search_faiss_index(query_embedding, faiss_index, k=k)

print(f"Query: {query}")
print("Top hotel with similar reviews using FAISS:")
for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    print(f"Review: {df_paris.iloc[idx]['review_text']}")
    # Note: With normalized embeddings and IndexFlatIP, the distance is the inner product,
    # which is equal to the cosine similarity.
    print(f"Cosine Similarity: {distance:.4f}")
    print()

In [ ]:
# Format the retrieval results as structured JSON for the LLM prompt.

import json

def search_hotels_by_query(query, model, faiss_index, df, k=25):
    """
    Performs a semantic search for hotels based on a query using a FAISS index.

    Args:
        query (str): The query string.
        model (SentenceTransformer): The pre-trained sentence transformer model.
        faiss_index (faiss.IndexFlatIP): The FAISS index object.
        df (pd.DataFrame): The DataFrame containing hotel data.
        k (int): The number of nearest neighbors to retrieve.

    Returns:
        str: A JSON string containing the query and the top k search results.
    """
    query_embedding = get_embeddings([query], model)
    distances, indices = search_faiss_index(query_embedding, faiss_index, k=k)

    results = []
    for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
        results.append({
            "rank": i,
            "hotel_name": df.iloc[idx]['hotel_name'],
            "review_text": df.iloc[idx]['review_text'],
            "cosine_similarity": float(distance) # Convert float32 to standard float
        })

    output = {
        "query": query,
        "top_results": results }

    return results

# Example usage:
# Assuming 'model', 'faiss_index', and 'df_paris' are already defined from the preceding code.
# query = "Hotel with a view of the Eiffel tower."
# json_output = search_hotels_by_query(query, model, faiss_index, df_paris, k=25)



In [ ]:
query = "Hotel with a view of the Eiffel tower."
json_output = search_hotels_by_query(query, model, faiss_index, df_paris, k=25)

In [ ]:
json_output

# 4. Full RAG: retrieve → augment → generate

We stream the answer from an OpenRouter-hosted LLM (default: `qwen/qwen3-8b`).


In [ ]:
# Retrieve API key securely from Colab user data
from google.colab import userdata
# Retrieve the value of a saved environment variable named 'OPEN_ROUTER_API_KEY'.
OPEN_ROUTER_API_KEY = userdata.get('OPEN_ROUTER_API_KEY')



# Initialize the OpenAI-compatible client, but point it to OpenRouter's API instead of OpenAI's
# OpenRouter is a gateway to multiple LLMs like GPT, Claude, Mistral, and others, through one unified API
from openai import OpenAI
open_router_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",  # Set the API endpoint to OpenRouter (not OpenAI)
    api_key=OPEN_ROUTER_API_KEY               # Use your OpenRouter API key for authentication
)


In [ ]:
# Combine retrieval output and the user query into a grounded RAG prompt.

# Define a function that uses a language model to generate an answer based on a user's query
def generate_answer(query):
    # Build the prompt that will be sent to the LLM
    # The prompt includes:
    # - Instructions to clean and format the answer
    # - The user's original query
    # - The context retrieved from Qdrant (via semantic search)

    json_output = search_hotels_by_query(query, model, faiss_index, df_paris, k=25)
    prompt = f"""
    Based on the following query from a user, please generate a small answer
    focusing on the original query and the response given. The answer should be paragraphs.
    Remove the special characters and (/n), make the output clean and long.
    Please cite source for each part as [1][2].
    Just start with the answer, no need to give any salutations.

    ###########
    query:
    "{query}"

    ########

    context:
    "{json_output}"
    #####

    Return in Markdown format.
    """

    # Send the prompt to the LLM using streaming mode
    # This allows the response to be received in real-time, piece by piece
    stream = open_router_client.chat.completions.create(
        model="qwen/qwen3-8b",  # Model to use (can be any OpenAI-compatible model, change the model here as needed)
        messages=[
            {
                "role": "user",
                "content": prompt,
            },
        ],
        stream=True,  # Enable streaming so we get partial output as it generates
    )

    # Initialize a variable to hold the full response
    output_text = ""

    # Iterate through the streaming response chunks
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            output_text += content  # Append new content to the full output
            print(content, end="")  # Print each chunk live as it's received

    # Return both the final answer and the context used (for reference or display)
    return output_text,json_output


In [ ]:
query = "Hotels with a view of the Eiffel Tower"
response,sources = generate_answer(query)

# 5. Spread to all cities — and add metadata filtering with Qdrant

FAISS gave us fast vector search. But it doesn't know that *some hotels are in Istanbul, others in Paris*. To filter by city we'd have to keep parallel arrays in sync ourselves.

**Qdrant** stores `{vector, payload}` per point and supports server-side filters. That's the production shape.


In [ ]:
dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")
df_all=pd.DataFrame(dataset['train'])
df_all.drop_duplicates()
reviews = df_all['review_text'].tolist()

In [ ]:
# Install and initialize an in-memory Qdrant instance.

# Import necessary modules from the Qdrant client library
# Qdrant is a vector database that allows you to store and search high-dimensional vector embeddings efficiently
!pip install qdrant_client

In [ ]:

from qdrant_client import QdrantClient, models

# Create a new Qdrant client instance using in-memory storage
# ":memory:" means the data will be stored temporarily in RAM (not saved to disk)
# Useful for testing or prototyping — everything is wiped when the program ends
client = QdrantClient(":memory:")

# Display the size (number of dimensions) of the text embeddings we generated earlier
# This is important because Qdrant needs to know the exact size of each vector to create a collection
text_embeddings_size = 768

In [ ]:
df_all.shape

In [ ]:
model

In [ ]:
df_all.locality.value_counts()

In [ ]:
# Drop rows with missing review text and embed the remaining reviews.

# Filter out rows with None review_text before converting to list
reviews_df = df_all.dropna(subset=['review_text'])
reviews = reviews_df['review_text'].tolist()



review_embeddings = get_embeddings(reviews, model)

In [ ]:
# Create a fresh Qdrant collection for hotel review vectors.


try:
    # Define the name of the collection we want to manage in Qdrant.
    # A collection in Qdrant is similar to a table in traditional databases —
    # it stores a group of vectors and their associated metadata.
    collection_name="hotel_reviews"

    # Check whether the collection already exists in Qdrant.
    # This avoids attempting to create a collection with a name that's already taken.
    if client.collection_exists(collection_name):
        # If the collection already exists, delete it to ensure we're starting fresh.
        # This is useful when we want to reset the state (e.g., during development or re-indexing).
        client.delete_collection(collection_name=collection_name)

        # Output a message confirming the collection was deleted successfully.
        print(f"Collection '{collection_name}' deleted successfully.")

    # Proceed to create a new collection regardless of whether it was previously deleted or not.
    # This ensures we always end up with a clean, newly-created collection.
    client.create_collection(
        collection_name=collection_name,  # The name of the new collection being created

        # Configure how vectors will be stored in this collection.
        # This includes the dimensionality (size) and the distance metric used for similarity.
        vectors_config=models.VectorParams(
            size=text_embeddings_size,       # The number of dimensions in each vector.
                                             # Must match the output size of your embedding model.
            distance=models.Distance.COSINE  # The distance function used for comparing vectors.
                                             # COSINE is commonly used for text embeddings as it measures angular similarity.
        ),
    )

    # Print a confirmation that the collection was created successfully.
    print(f"Collection '{collection_name}' created successfully.")

except Exception as e:
    # If any error occurs during the process (e.g., connection issues, invalid parameters),
    # it will be caught here and the error message will be printed.
    print(f"An error occurred while setting up the collection: {e}")


In [ ]:
# Upload review vectors plus metadata payloads into Qdrant.

# Prepare the data for uploading to Qdrant
# We need to create "points", where each point contains:
# 1. An `id` (unique identifier for the point/review)
# 2. A `vector` (the embedding of the review text)
# 3. `payload` (metadata associated with the vector, like hotel name, city, etc.)


# Example usage (assuming 'reviews' and 'model' are defined from previous code):

points_to_upload = [
    models.PointStruct(
        id=i,  # Use the index as the unique ID
        vector=review_embeddings[i].tolist(),  # Convert numpy array to list for Qdrant
        payload={
            "hotel_name": reviews_df.iloc[i]['hotel_name'],
            "review_text": reviews_df.iloc[i]['review_text'],
            "locality": reviews_df.iloc[i]['locality'], # Include locality for filtering
        },
    )
    for i in range(len(review_embeddings))
]

# Upload the created points to the Qdrant collection in batches
client.upsert(
    collection_name="hotel_reviews",
    wait=True,  # Wait until the operation is completed
    points=points_to_upload,
)

# Function to search Qdrant with an optional city filter
def search_qdrant_with_filter(query, model, client, city=None, k=10):
    """
    Performs a semantic search on the Qdrant collection with an optional city filter.

    Args:
        query (str): The query string.
        model (SentenceTransformer): The pre-trained sentence transformer model.
        client (QdrantClient): The Qdrant client instance.
        city (str, optional): The name of the city to filter by. Defaults to None.
        k (int): The number of nearest neighbors to retrieve.

    Returns:
        list: A list of search results from Qdrant.
    """
    # Generate the embedding for the query
    query_embedding = model.encode(query, show_progress_bar=False).tolist()

    # Define the filter
    # If a city is specified, create a filter that matches the 'locality' field
    # Otherwise, the filter is None (no filtering)
    query_filter = None
    if city:
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="locality",
                    match=models.MatchValue(value=city)
                )
            ]
        )

    # Perform the search


    text_hits = client.query_points(
    collection_name="hotel_reviews",  # The name of the collection where vectors were stored
    query=query_embedding,
    query_filter=query_filter,   # The query vector — what we want to find similar results to
    limit=k,                             # Limit the number of results to 3 most relevant chunks
    with_payload=True, # Include the payload (metadata) in the results
).points                                 # Extract only the list of matching points (each with vector + payload)



    return text_hits





### City-filtered semantic search


In [ ]:
# Search for hotels with a view of the Eiffel Tower specifically in Paris
query = "Amazing hotel close to everythings"
city_filter = "Istanbul"
text_hits = search_qdrant_with_filter(query, model, client, city=city_filter, k=10)

In [ ]:
text_hits

In [ ]:
# Search for hotels with a generic query and city name
query = "Amazing hotel close to everythings"
city_filter = "Istanbul"
qdrant_results_city = search_qdrant_with_filter(query, model, client, city=city_filter, k=10)

print(f"Query: {query} in {city_filter}")
print("Top hotel reviews using Qdrant with city filter:")
for i, result in enumerate(qdrant_results_city, 1):
    print(f"{i}. Hotel Name: {result.payload['hotel_name']}")
    print(f"   Review: {result.payload['review_text']}")
    print(f"   Score (Cosine Similarity): {result.score:.4f}")
    print(f"   Locality: {result.payload['locality']}")
    print()

# Search for hotels with a view of the Eiffel Tower without a specific city filter (searches across all data loaded into the collection)
query = "Hotel close to Hagia Sofia."
qdrant_results_all = search_qdrant_with_filter(query, model, client, city=None, k=10)

print(f"\nQuery: {query} (without city filter)")
print("Top hotel reviews using Qdrant (no filter):")
for i, result in enumerate(qdrant_results_all, 1):
    print(f"{i}. Hotel Name: {result.payload['hotel_name']}")
    print(f"   Review: {result.payload['review_text']}")
    print(f"   Score (Cosine Similarity): {result.score:.4f}")
    print(f"   Locality: {result.payload['locality']}")
    print()




### Full RAG with city filter

Same retrieve → augment → generate flow, but the retrieval is now scoped to one city.


In [ ]:
# Use Qdrant retrieval results as context for the final answer.

# Modify the generate_answer function to use Qdrant search
def generate_answer_qdrant(query, city=None):
    """
    Generates an answer using a language model based on a user's query
    and context retrieved from Qdrant, with an optional city filter.

    Args:
        query (str): The user's query string.
        city (str, optional): The name of the city to filter Qdrant search results by. Defaults to None.

    Returns:
        tuple: A tuple containing:
            - output_text (str): The generated answer from the LLM.
            - sources (list): The search results from Qdrant used as context.
    """
    # Use the new search_qdrant_with_filter function
    qdrant_results = search_qdrant_with_filter(query, model, client, city=city, k=25)

    # Format the Qdrant results into a string that the LLM can understand
    # You might want to adjust this formatting based on the LLM you use
    context_string = ""
    for i, result in enumerate(qdrant_results):
        context_string += f"Source {i+1}:\n"
        context_string += f"Hotel: {result.payload.get('hotel_name', 'N/A')}\n"
        context_string += f"Review: {result.payload.get('review_text', 'N/A')}\n"
        context_string += f"Locality: {result.payload.get('locality', 'N/A')}\n"
        context_string += f"Similarity Score: {result.score:.4f}\n\n"


    # Build the prompt for the LLM
    prompt = f"""
    Based on the following query from a user and the provided context from hotel reviews,
    please generate a concise answer summarizing the relevant information.
    Focus on addressing the user's query using details found in the reviews.
    Cite the sources using numerical references like [1], [2], etc., corresponding to the "Source #" in the context.
    Format the output as a few paragraphs.
    Remove any special characters like (/n) and ensure the output is clean.
    Begin directly with the answer.

    ###########
    query:
    "{query}"

    ########

    context:
    "{context_string}"
    #####

    Return in Markdown format.
    """

    # Send the prompt to the LLM using streaming mode
    stream = open_router_client.chat.completions.create(
        model="qwen/qwen3-8b",  # Model to use (change as needed)
        messages=[
            {
                "role": "user",
                "content": prompt,
            },
        ],
        stream=True,  # Enable streaming
    )

    # Initialize a variable to hold the full response
    output_text = ""

    # Iterate through the streaming response chunks and print them
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            output_text += content
            print(content, end="")

    # Return the generated text and the Qdrant search results
    return output_text, qdrant_results

In [ ]:
# Query for hotels with generic example
query= "Amazing hotel close to everything"
city_filter = "Istanbul"
response_paris, sources_paris = generate_answer_qdrant(query, city=city_filter)

print("\n--- Generated Answer (City Filter) ---")
# The response was already printed chunk by chunk inside the function.
# You can print the final output_text here again if needed:
# print(response_paris)

print("\n--- Sources Used (City Filter) ---")
# Print details about the sources (Qdrant results) that were used
for i, result in enumerate(sources_paris):
    print(f"Source {i+1}: Hotel: {result.payload.get('hotel_name', 'N/A')}, Locality: {result.payload.get('locality', 'N/A')}, Score: {result.score:.4f}")

print("\n" + "="*50 + "\n") # Separator


## What's next

Notebook **6b** repeats this pipeline on a very different corpus — research-paper abstracts — adding:

- Document chunking (papers can be longer than the embedding model's context window)
- HuggingFace `transformers` directly (skipping `sentence-transformers`)
- Persisting the Qdrant collection to disk
